# Bài tập: Các kỹ thuật mã hóa văn bản
## Dữ liệu: Biển báo giao thông Việt Nam

Notebook này minh họa 4 kỹ thuật mã hóa văn bản:
1. **One-hot Encoding**: Mã hóa mỗi từ thành vector nhị phân
2. **Count Vectorizing**: Đếm số lần xuất hiện của từ trong văn bản
3. **N-grams**: Trích xuất các chuỗi n từ liên tiếp
4. **Co-occurrence Matrix**: Ma trận đồng xuất hiện của các từ

In [1]:
# Import thư viện cần thiết
import json
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình hiển thị
plt.rcParams['font.family'] = 'DejaVu Sans'
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

ModuleNotFoundError: No module named 'pandas'

## 1. Đọc và chuẩn bị dữ liệu

In [ ]:
# Đọc dữ liệu đã được xử lý
with open('../data/processed/traffic_signs_processed.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Lấy danh sách mô tả đã xử lý
descriptions = [sign['description_processed'] for sign in data['signs']]
sign_names = [sign['name'] for sign in data['signs']]

print(f"Tổng số biển báo: {len(descriptions)}")
print(f"\nVí dụ 10 mô tả đầu tiên:")
for i, (name, desc) in enumerate(zip(sign_names[:10], descriptions[:10])):
    print(f"{i+1}. {name}: {desc}")

## 2. One-Hot Encoding

**Khái niệm**: Mã hóa mỗi từ thành một vector nhị phân có độ dài bằng kích thước từ vựng, với giá trị 1 tại vị trí tương ứng với từ đó và 0 ở các vị trí khác.

**Ví dụ**: 
- Từ vựng: ["cấm", "xe", "đường"]
- "cấm" → [1, 0, 0]
- "xe" → [0, 1, 0]
- "đường" → [0, 0, 1]

In [ ]:
# Tạo từ vựng từ tất cả các mô tả
all_words = []
for desc in descriptions:
    all_words.extend(desc.split())

# Lấy từ vựng duy nhất và sắp xếp
vocabulary = sorted(set(all_words))
word_to_index = {word: idx for idx, word in enumerate(vocabulary)}

print(f"Kích thước từ vựng: {len(vocabulary)}")
print(f"\n20 từ đầu tiên trong từ vựng: {vocabulary[:20]}")

In [ ]:
def one_hot_encode_word(word, word_to_index):
    """Mã hóa one-hot cho một từ"""
    vector = [0] * len(word_to_index)
    if word in word_to_index:
        vector[word_to_index[word]] = 1
    return vector

def one_hot_encode_sentence(sentence, word_to_index):
    """Mã hóa one-hot cho một câu (trung bình các vector từ)"""
    words = sentence.split()
    vectors = [one_hot_encode_word(word, word_to_index) for word in words]
    # Trung bình các vector
    if vectors:
        return np.mean(vectors, axis=0)
    return [0] * len(word_to_index)

# Ví dụ: Mã hóa một vài từ
example_words = ["cấm", "xe", "đường", "rẽ", "trái"]
print("One-Hot Encoding cho các từ:\n")
for word in example_words:
    if word in word_to_index:
        vector = one_hot_encode_word(word, word_to_index)
        print(f"{word:10} → vị trí {word_to_index[word]} = 1 (vector có {len(vector)} chiều)")
        # Hiển thị một phần vector
        print(f"            Vector: {vector[:10]}... (chỉ hiển thị 10 giá trị đầu)\n")

In [ ]:
# Mã hóa một vài câu mẫu
sample_descriptions = descriptions[:5]
one_hot_matrix = []

print("One-Hot Encoding cho câu (trung bình vector):\n")
for i, desc in enumerate(sample_descriptions):
    vector = one_hot_encode_sentence(desc, word_to_index)
    one_hot_matrix.append(vector)
    print(f"{sign_names[i]}: {desc}")
    print(f"Vector (10 giá trị đầu): {vector[:10]}")
    print(f"Tổng: {sum(vector):.2f}\n")

# Chuyển thành DataFrame
df_one_hot = pd.DataFrame(one_hot_matrix, 
                          columns=vocabulary,
                          index=[sign_names[i] for i in range(5)])
print("\nMa trận One-Hot (5 câu đầu, 15 từ đầu):")
print(df_one_hot.iloc[:, :15])

## 3. Count Vectorizing (Bag of Words)

**Khái niệm**: Đếm số lần xuất hiện của mỗi từ trong văn bản, tạo thành một vector đặc trưng.

**Ví dụ**:
- Câu 1: "cấm xe ô tô" → {cấm: 1, xe: 1, ô: 1, tô: 1}
- Câu 2: "cấm xe máy" → {cấm: 1, xe: 1, máy: 1}

In [ ]:
# Sử dụng CountVectorizer từ sklearn
count_vectorizer = CountVectorizer()
count_matrix = count_vectorizer.fit_transform(descriptions)

# Lấy tên các đặc trưng (từ)
feature_names = count_vectorizer.get_feature_names_out()

print(f"Kích thước ma trận Count Vector: {count_matrix.shape}")
print(f"(Số mẫu: {count_matrix.shape[0]}, Số từ vựng: {count_matrix.shape[1]})")

# Chuyển thành DataFrame để dễ xem
df_count = pd.DataFrame(count_matrix.toarray()[:10], 
                        columns=feature_names,
                        index=sign_names[:10])

print("\nCount Vector cho 10 biển báo đầu tiên (15 từ đầu):")
print(df_count.iloc[:, :15])

In [ ]:
# Thống kê các từ xuất hiện nhiều nhất
word_counts = np.asarray(count_matrix.sum(axis=0)).flatten()
word_freq = pd.DataFrame({'word': feature_names, 'count': word_counts})
word_freq = word_freq.sort_values('count', ascending=False)

print("\n20 từ xuất hiện nhiều nhất:")
print(word_freq.head(20))

# Vẽ biểu đồ
plt.figure(figsize=(12, 6))
plt.bar(range(20), word_freq.head(20)['count'])
plt.xticks(range(20), word_freq.head(20)['word'], rotation=45, ha='right')
plt.xlabel('Từ')
plt.ylabel('Số lần xuất hiện')
plt.title('20 từ xuất hiện nhiều nhất trong mô tả biển báo')
plt.tight_layout()
plt.show()

## 4. N-grams

**Khái niệm**: Trích xuất các chuỗi n từ liên tiếp trong văn bản.

- **Unigram** (1-gram): Các từ đơn lẻ ["cấm", "xe", "ô", "tô"]
- **Bigram** (2-gram): Cặp từ liên tiếp ["cấm xe", "xe ô", "ô tô"]
- **Trigram** (3-gram): Bộ ba từ liên tiếp ["cấm xe ô", "xe ô tô"]

In [ ]:
# Tạo Bigrams (2-grams)
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2), max_features=100)
bigram_matrix = bigram_vectorizer.fit_transform(descriptions)
bigram_features = bigram_vectorizer.get_feature_names_out()

print("=" * 60)
print("BIGRAMS (2-grams)")
print("=" * 60)
print(f"Tổng số bigrams: {len(bigram_features)}")

# Thống kê bigrams phổ biến nhất
bigram_counts = np.asarray(bigram_matrix.sum(axis=0)).flatten()
bigram_freq = pd.DataFrame({'bigram': bigram_features, 'count': bigram_counts})
bigram_freq = bigram_freq.sort_values('count', ascending=False)

print("\n20 bigrams xuất hiện nhiều nhất:")
print(bigram_freq.head(20))

In [ ]:
# Tạo Trigrams (3-grams)
trigram_vectorizer = CountVectorizer(ngram_range=(3, 3), max_features=100)
trigram_matrix = trigram_vectorizer.fit_transform(descriptions)
trigram_features = trigram_vectorizer.get_feature_names_out()

print("=" * 60)
print("TRIGRAMS (3-grams)")
print("=" * 60)
print(f"Tổng số trigrams: {len(trigram_features)}")

# Thống kê trigrams phổ biến nhất
trigram_counts = np.asarray(trigram_matrix.sum(axis=0)).flatten()
trigram_freq = pd.DataFrame({'trigram': trigram_features, 'count': trigram_counts})
trigram_freq = trigram_freq.sort_values('count', ascending=False)

print("\n20 trigrams xuất hiện nhiều nhất:")
print(trigram_freq.head(20))

In [ ]:
# Kết hợp các n-grams (1-3)
combined_vectorizer = CountVectorizer(ngram_range=(1, 3), max_features=50)
combined_matrix = combined_vectorizer.fit_transform(descriptions)
combined_features = combined_vectorizer.get_feature_names_out()

print("=" * 60)
print("KẾT HỢP UNIGRAMS, BIGRAMS, TRIGRAMS")
print("=" * 60)

# Top n-grams
combined_counts = np.asarray(combined_matrix.sum(axis=0)).flatten()
combined_freq = pd.DataFrame({'ngram': combined_features, 'count': combined_counts})
combined_freq = combined_freq.sort_values('count', ascending=False)

print("\nTop 30 n-grams (kết hợp 1-3 grams):")
print(combined_freq.head(30))

# Vẽ biểu đồ
plt.figure(figsize=(14, 6))
top_n = 25
plt.barh(range(top_n), combined_freq.head(top_n)['count'])
plt.yticks(range(top_n), combined_freq.head(top_n)['ngram'])
plt.xlabel('Số lần xuất hiện')
plt.ylabel('N-gram')
plt.title(f'Top {top_n} N-grams phổ biến nhất')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Co-occurrence Matrix (Ma trận Đồng xuất hiện)

**Khái niệm**: Ma trận thể hiện số lần hai từ xuất hiện cùng nhau trong cùng một ngữ cảnh (cửa sổ từ, câu, hoặc văn bản).

**Ứng dụng**: Phát hiện mối quan hệ giữa các từ, tìm từ đồng nghĩa hoặc các từ thường đi kèm nhau.

In [ ]:
def build_cooccurrence_matrix(descriptions, window_size=2, top_words=30):
    """
    Xây dựng ma trận đồng xuất hiện
    
    Args:
        descriptions: Danh sách các câu/văn bản
        window_size: Kích thước cửa sổ ngữ cảnh
        top_words: Số từ phổ biến nhất để xây dựng ma trận
    """
    # Tìm các từ phổ biến nhất
    word_counter = Counter()
    for desc in descriptions:
        word_counter.update(desc.split())
    
    # Lấy top words
    top_word_list = [word for word, _ in word_counter.most_common(top_words)]
    word_to_idx = {word: idx for idx, word in enumerate(top_word_list)}
    
    # Khởi tạo ma trận đồng xuất hiện
    cooc_matrix = np.zeros((top_words, top_words))
    
    # Đếm đồng xuất hiện
    for desc in descriptions:
        words = desc.split()
        for i, word in enumerate(words):
            if word not in word_to_idx:
                continue
            
            # Xem xét các từ trong cửa sổ ngữ cảnh
            start = max(0, i - window_size)
            end = min(len(words), i + window_size + 1)
            
            for j in range(start, end):
                if i != j and words[j] in word_to_idx:
                    cooc_matrix[word_to_idx[word]][word_to_idx[words[j]]] += 1
    
    return cooc_matrix, top_word_list

# Xây dựng ma trận với window_size=2
print("Xây dựng ma trận đồng xuất hiện...")
cooc_matrix, top_words = build_cooccurrence_matrix(descriptions, window_size=2, top_words=30)

print(f"\nKích thước ma trận: {cooc_matrix.shape}")
print(f"Số từ được phân tích: {len(top_words)}")
print(f"\nTop 30 từ: {top_words}")

In [ ]:
# Tạo DataFrame cho ma trận
df_cooc = pd.DataFrame(cooc_matrix, index=top_words, columns=top_words)

print("Ma trận đồng xuất hiện (15x15 đầu tiên):")
print(df_cooc.iloc[:15, :15])

In [ ]:
# Vẽ heatmap cho ma trận đồng xuất hiện
plt.figure(figsize=(16, 14))
sns.heatmap(df_cooc, 
            annot=False,  # Không hiển thị số vì quá nhiều
            cmap='YlOrRd',
            square=True,
            cbar_kws={'label': 'Số lần đồng xuất hiện'})
plt.title('Ma trận đồng xuất hiện của Top 30 từ (Window Size = 2)', fontsize=14, pad=20)
plt.xlabel('Từ', fontsize=12)
plt.ylabel('Từ', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Tìm các từ thường đi kèm với một từ cụ thể
def find_related_words(target_word, cooc_matrix, word_list, top_n=10):
    """Tìm các từ thường đi kèm với từ mục tiêu"""
    if target_word not in word_list:
        print(f"Từ '{target_word}' không có trong danh sách")
        return
    
    idx = word_list.index(target_word)
    cooc_scores = cooc_matrix[idx]
    
    # Sắp xếp và lấy top từ
    related_indices = np.argsort(cooc_scores)[::-1][:top_n]
    
    print(f"\nTop {top_n} từ thường đi kèm với '{target_word}':")
    print("-" * 50)
    for i, idx in enumerate(related_indices, 1):
        if cooc_scores[idx] > 0:
            print(f"{i:2d}. {word_list[idx]:15s} - {int(cooc_scores[idx])} lần")

# Phân tích một số từ quan trọng
important_words = ['cấm', 'xe', 'đường', 'rẽ', 'tốc']

for word in important_words:
    find_related_words(word, cooc_matrix, top_words, top_n=8)

In [ ]:
# Tạo ma trận với window_size khác nhau để so sánh
print("So sánh với các kích thước cửa sổ khác nhau:\n")

for window in [1, 2, 3, 5]:
    cooc_temp, _ = build_cooccurrence_matrix(descriptions, window_size=window, top_words=20)
    total_cooccurrences = int(np.sum(cooc_temp))
    print(f"Window size = {window}: Tổng số đồng xuất hiện = {total_cooccurrences:,}")

## 6. So sánh và Kết luận

### So sánh các phương pháp:

| Phương pháp | Ưu điểm | Nhược điểm | Ứng dụng |
|-------------|---------|------------|----------|
| **One-Hot Encoding** | Đơn giản, dễ hiểu | Ma trận thưa, không thể hiện ngữ nghĩa | Phân loại văn bản cơ bản |
| **Count Vectorizing** | Bảo toàn tần suất từ, hiệu quả | Không xét thứ tự từ, bị ảnh hưởng bởi độ dài văn bản | Text classification, clustering |
| **N-grams** | Giữ ngữ cảnh cục bộ, phát hiện cụm từ | Tăng số chiều nhanh, sparse matrix | Language modeling, phrase detection |
| **Co-occurrence Matrix** | Thể hiện quan hệ giữa các từ | Phức tạp tính toán, cần nhiều bộ nhớ | Word similarity, semantic analysis |

## 7. Bài tập thực hành (Tự làm)

1. **One-Hot Encoding**: Thử mã hóa một câu tự đặt và kiểm tra kết quả
2. **Count Vectorizing**: Sử dụng TF-IDF thay vì Count để cải thiện kết quả
3. **N-grams**: Tìm các 4-grams và 5-grams phổ biến nhất
4. **Co-occurrence**: Thay đổi window_size và phân tích sự khác biệt
5. **Thực hành**: Kết hợp các phương pháp để xây dựng mô hình phân loại biển báo